# 🎬 Movie Recommender System
---
### 📌 Project Overview
This notebook builds a **Content-Based Movie Recommender System** using the TMDB 5000 Movies dataset.

The idea is simple — every movie is converted into a **bag of words (tags)** using its plot, genres, keywords, cast and director. Then we use **Cosine Similarity** to find which movies are closest to each other in this tag space.

---
### 🗂️ Dataset
- `tmdb_5000_movies.csv` → movie metadata (genres, keywords, overview, budget, etc.)
- `tmdb_5000_credits.csv` → cast & crew info per movie

---
### 🧠 Technique Used
```
Raw Text Columns
      ↓
Parse JSON strings → clean lists
      ↓
Combine into one 'tags' string per movie
      ↓
CountVectorizer → Bag of Words vectors
      ↓
Cosine Similarity Matrix
      ↓
recommend(movie) → Top 5 similar movies
```

---
## 📦 Step 1 — Import Libraries

### 🔍 Why these libraries?

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations on arrays |
| `pandas` | Load CSV files, DataFrames, data manipulation |
| `ast` | Safely parse stringified Python lists/dicts from columns |
| `pickle` | Save and load our trained model files |
| `CountVectorizer` | Convert text → numerical word-count vectors |
| `cosine_similarity` | Measure similarity between two vectors |

In [34]:
import numpy as np
import pandas as pd
import ast
import pickle

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

---
## 📂 Step 2 — Load the Dataset

### 🔍 What are these files?

- **movies CSV** → 4803 rows, 20 columns — movie metadata like budget, genres, keywords, overview, popularity, release date, etc.
- **credits CSV** → 4803 rows, 4 columns — movie_id, title, cast (JSON), crew (JSON)

> 💡 Both files are available on Kaggle under `tmdb-movie-metadata`

In [39]:
movies  = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print("Movies shape :", movies.shape)
print("Credits shape:", credits.shape)

Movies shape : (4803, 20)
Credits shape: (4803, 4)


In [41]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [45]:
credits.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


---
## 🔗 Step 3 — Merge Both DataFrames

### 🔍 Why merge?
The cast/crew info lives in the credits file, while genres/keywords are in the movies file.  
We merge them on the common `title` column so all info is in one place.

> ⚠️ After merge, shape becomes `(4803, 23)` — 20 original columns + 3 from credits (movie_id, cast, crew)

In [48]:
movies = movies.merge(credits, on='title')
print("Shape after merge:", movies.shape)

Shape after merge: (4809, 23)


---
## ✂️ Step 4 — Select Only Relevant Columns

### 🔍 Which columns do we keep and why?

| Column | Why we need it |
|---|---|
| `movie_id` | Unique ID → used to fetch poster from TMDB API in the app |
| `title` | Movie name → shown in the UI |
| `overview` | Plot summary → most important text signal |
| `genres` | Genre list → Action, Adventure, etc. |
| `keywords` | Keywords list → 'space war', 'based on novel', etc. |
| `cast` | Actor names → top 3 are strong similarity signals |
| `crew` | Director only → director style matters a lot |

> 🗑️ We drop columns like `budget`, `revenue`, `runtime`, `popularity` etc. because they don't help with **content similarity**.

In [51]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


---
## 🧹 Step 5 — Handle Missing Values

### 🔍 Why drop nulls?
A few movies have missing overviews or empty genre lists.  
If we keep them, the tag string will be incomplete and produce bad recommendations.  
Safest approach here = **drop those rows** (very few, ~3-4 rows).

In [54]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [56]:
movies.dropna(inplace=True)
print("Shape after dropna:", movies.shape)

Shape after dropna: (4806, 7)


---
## 🔧 Step 6 — Write Helper Functions

### 🔍 The Problem with genres/keywords/cast/crew columns
Right now these columns look like this:
```
'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]'
```
It's a **string**, not a Python list. We need to:
1. Parse it back into a real Python list using `ast.literal_eval()`
2. Extract just the `name` values

---

### 🛠️ Function 1 — `convert()` → Extract ALL names
Used for `genres` and `keywords`

In [59]:
def convert(text):
    """
    Extracts ALL 'name' values from a stringified list of dicts.

    Input  : '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]'
    Output : ['Action', 'Adventure']
    """
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

### 🛠️ Function 2 — `convert3()` → Extract only TOP 3 names
Used for `cast`

> 💡 Why only 3 actors? — Having 20+ actors per movie creates too much noise. The top 3 billed actors are the most recognizable and most useful for matching.

In [62]:
def convert3(text):
    """
    Extracts only the first 3 'name' values from a stringified list of dicts.

    Input  : '[{"name": "Sam Worthington"}, {"name": "Zoe Saldana"}, ...]'
    Output : ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']
    """
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
        counter += 1
    return L

### 🛠️ Function 3 — `fetch_director()` → Extract Director from crew

> 💡 Why only the director? — The crew column has 100s of people (editors, sound designers, VFX artists etc.). The **director** is the single biggest creative signature of a film — James Cameron movies feel like James Cameron movies.

In [65]:
def fetch_director(text):
    """
    Scans the crew list and returns only the Director(s).

    Input  : '[{"job": "Director", "name": "James Cameron"}, {"job": "Producer", ...}]'
    Output : ['James Cameron']
    """
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

### 🛠️ Function 4 — `collapse()` → Remove spaces from names

> 💡 Why collapse spaces? — `CountVectorizer` splits on spaces. So `'Sam Worthington'` becomes two separate tokens: `'sam'` and `'worthington'`. That means 'Sam Raimi' and 'Sam Worthington' would incorrectly share the token `'sam'`. By joining: `'SamWorthington'` stays one unique token and never false-matches.

In [68]:
def collapse(L):
    """
    Removes spaces from each string in a list so multi-word items
    become single tokens for CountVectorizer.

    Input  : ['Sam Worthington', 'Zoe Saldana', 'Science Fiction']
    Output : ['SamWorthington', 'ZoeSaldana', 'ScienceFiction']
    """
    L1 = []
    for i in L:
        L1.append(i.replace(" ", ""))
    return L1

---
## 🏗️ Step 7 — Apply All Transformations

### 🔍 What happens to each column?

| Column | Before | After |
|---|---|---|
| `genres` | `'[{"id":28,"name":"Action"},...]'` (string) | `['Action', 'Adventure']` (list) |
| `keywords` | stringified JSON | clean list of keyword strings |
| `cast` | stringified JSON | top 3 actor names as list |
| `crew` | stringified JSON | `['JamesCameron']` — director only |
| `overview` | `'In the 22nd century...'` (string) | `['In', 'the', '22nd', ...]` (list of words) |

In [71]:
# genres → all genres as list
movies['genres'] = movies['genres'].apply(convert)

# keywords → all keywords as list
movies['keywords'] = movies['keywords'].apply(convert)

# cast → parse all, then slice to top 3
movies['cast'] = movies['cast'].apply(convert)
movies['cast'] = movies['cast'].apply(lambda x: x[0:3])

# crew → director only
movies['crew'] = movies['crew'].apply(fetch_director)

# overview → split string into list of words
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# collapse spaces from all list columns
movies['cast']     = movies['cast'].apply(collapse)
movies['crew']     = movies['crew'].apply(collapse)
movies['genres']   = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


---
## 🏷️ Step 8 — Create the `tags` Column

### 🔍 What is a 'tag'?
We combine all 5 signal columns into one flat list, then join with spaces into a single string.

```
tags = overview + genres + keywords + cast + crew
```

**Example for Avatar:**
```
'in the 22nd century a paraplegic marine ... Action Adventure ScienceFiction
 cultureclash future spacewar ... SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'
```

> 💡 This single string now represents the entire "identity" of the movie. Two movies with similar tags will have similar vectors → high cosine similarity → recommended together.

In [76]:
# Combine all columns into one list
movies['tags'] = (movies['overview']
                  + movies['genres']
                  + movies['keywords']
                  + movies['cast']
                  + movies['crew'])

# Keep only what we need
new = movies[['movie_id', 'title', 'tags']].copy()  

# Convert list → single space-separated string
new['tags'] = new['tags'].apply(lambda x: " ".join(x))

# Lowercase for uniformity
new['tags'] = new['tags'].apply(lambda x: x.lower())

new.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


---
## 🔢 Step 9 — Vectorization using CountVectorizer

### 🔍 What does CountVectorizer do?
It converts each movie's tag string into a **numerical vector** where each position represents a word and the value is how many times that word appears.

```
"action adventure spaceship"  →  [1, 1, 0, 0, 0, 1, ...]
"action thriller crime"       →  [1, 0, 1, 1, 0, 0, ...]
                                   ↑action  ↑adventure  ↑crime ...
```

### 🔍 Why `max_features=5000`?
We keep only the **top 5000 most frequent** words across all movies. Very rare words (appear in only 1 movie) don't help with similarity and just add noise.

### 🔍 Why `stop_words='english'`?
Words like `the`, `a`, `is`, `and` appear in every movie's overview. They carry **zero useful information** for distinguishing movies — removing them makes the vectors much cleaner.

In [79]:
cv = CountVectorizer(max_features=5000, stop_words='english')

# fit_transform → learns vocabulary + converts all tags to vectors
# .toarray()    → converts sparse matrix to dense numpy array
vector = cv.fit_transform(new['tags']).toarray()

print("Vector shape:", vector.shape)
# (4806 movies, 5000 unique words) — each movie is a 5000-dimensional vector

Vector shape: (4806, 5000)


---
## 📐 Step 10 — Compute Cosine Similarity

### 🔍 Why Cosine Similarity and not Euclidean Distance?

Imagine two movies:
- Movie A has a 2-sentence overview → short vector
- Movie B has a 10-sentence overview → long vector

If they talk about similar things, **Euclidean distance** would say they're far apart (different magnitudes).  
**Cosine Similarity** only looks at the **angle** between vectors — length doesn't matter.  
So two similar movies will still get a high score regardless of overview length. ✅

```
cos(θ) = (A · B) / (|A| × |B|)

Range: 0 (completely different) → 1 (identical)
```

### 🔍 What does the similarity matrix look like?
`similarity[i][j]` = how similar movie `i` is to movie `j`  
It's a **square matrix** of shape `(num_movies × num_movies)`

In [82]:
similarity = cosine_similarity(vector)

print("Similarity matrix shape:", similarity.shape)
# e.g. (4806, 4806)

# Peek at Avatar's similarity scores with first 5 movies
print("\nAvatar similarity with first 5 movies:")
print(similarity[0][:5])

Similarity matrix shape: (4806, 4806)

Avatar similarity with first 5 movies:
[1.         0.08964215 0.06071767 0.03984095 0.18257419]


---
## 🎯 Step 11 — Build the `recommend()` Function

### 🔍 How does recommendation work?

```
Input movie: 'Gandhi'
     ↓
Find row index of 'Gandhi' in DataFrame → say index = 84
     ↓
similarity[84] → array of 4806 scores (one per movie)
     ↓
Sort by score descending → [(84, 1.0), (312, 0.43), (89, 0.38), ...]
     ↓
Skip index 0 (that's Gandhi itself, score = 1.0)
     ↓
Print titles at indices 1 to 5
```

In [85]:
def recommend(movie):
    """
    Prints the top 5 most similar movies to the given title.

    Args:
        movie (str): Exact movie title from the dataset.
    """
    # Step 1: Find the integer row index of this movie
    index = new[new['title'] == movie].index[0]

    # Step 2: Get all (index, similarity_score) pairs and sort by score desc
    distances = sorted(
        list(enumerate(similarity[index])),
        reverse=True,
        key=lambda x: x[1]
    )

    # Step 3: Print top 5 (skip index 0 which is the movie itself)
    print(f"\n🎬 Movies similar to '{movie}':\n")
    for i in distances[1:6]:
        print(" →", new.iloc[i[0]].title)

In [87]:
# 🧪 Test 1
recommend('Gandhi')


🎬 Movies similar to 'Gandhi':

 → Gandhi, My Father
 → The Wind That Shakes the Barley
 → A Passage to India
 → Guiana 1838
 → Ramanujan


In [89]:
# 🧪 Test 2
recommend('The Dark Knight Rises')


🎬 Movies similar to 'The Dark Knight Rises':

 → The Dark Knight
 → Batman Begins
 → Batman
 → Batman Returns
 → Batman


In [91]:
# 🧪 Test 3
recommend('Avatar')


🎬 Movies similar to 'Avatar':

 → Titan A.E.
 → Small Soldiers
 → Ender's Game
 → Aliens vs Predator: Requiem
 → Independence Day


---
## 💾 Step 12 — Save Model Files with Pickle

### 🔍 Why pickle?
The Streamlit app (`app.py`) needs both the DataFrame and the similarity matrix at runtime.

We have two options:
1. ❌ **Recompute everything each time** the app loads → takes 30–60 seconds, very bad UX
2. ✅ **Save to .pkl files** → load in milliseconds

Pickle serializes Python objects to binary files. We save:
- `movie_list.pkl` → the `new` DataFrame (movie_id, title, tags)
- `similarity.pkl` → the full cosine similarity matrix

> 📁 These files go into a `model/` folder which `app.py` reads from.

In [94]:
import os
os.makedirs('model', exist_ok=True)

pickle.dump(new, open('model/movie_list.pkl', 'wb'))
pickle.dump(similarity, open('model/similarity.pkl', 'wb'))

print("✅ Saved model/movie_list.pkl")
print("✅ Saved model/similarity.pkl")
print("\n🚀 Now run: streamlit run app.py")

✅ Saved model/movie_list.pkl
✅ Saved model/similarity.pkl

🚀 Now run: streamlit run app.py
